# GLPI Reserved Items — Bronze to Silver (snapshot dimension)

Purpose:
- Read GLPI ReservationItem data from Bronze (ingested with expand_dropdowns=true)
- Keep ONLY the latest ingestion_date partition (daily snapshot)
- Clean types + rename columns so it matches reservations join key: `reservation_item_id`
- Parse useful IDs from `links` (entity_id, room_id)
- Deduplicate by reservation_item_id (keep latest _ingest_bronze_ts)
- Write a Silver snapshot table for downstream Gold joins

## Data Access And Configuration

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import re

In [0]:
storage_account = "startupvillagedatalake"
scope_name = "kv-startupvillage"   

client_id = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=scope_name, key="tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
dbutils.fs.ls("abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/")

[FileInfo(path='abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/computers/', name='computers/', size=0, modificationTime=1772894299000),
 FileInfo(path='abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/reservations/', name='reservations/', size=0, modificationTime=1772894304000),
 FileInfo(path='abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/', name='reserveditems/', size=0, modificationTime=1776012069000),
 FileInfo(path='abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/tickets/', name='tickets/', size=0, modificationTime=1772894276000),
 FileInfo(path='abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/users/', name='users/', size=0, modificationTime=1772894292000)]

## Paths

In [0]:
bronze_path = "abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems"
silver_path = "abfss://silver@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems"

print("Bronze:", bronze_path)
print("Silver:", silver_path)

Bronze: abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems
Silver: abfss://silver@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems


## Read Bronze (latest ingestion_date only)

In [0]:
df_b = spark.read.format("delta").load(bronze_path)

latest_date = df_b.select(max(col("ingestion_date")).alias("d")).collect()[0]["d"]
if latest_date is None:
    raise Exception(f"No ingestion_date found in {bronze_path}")

df = df_b.filter(col("ingestion_date") == lit(latest_date))

print("Latest ingestion_date:", latest_date)
print("Bronze rows (latest):", df.count())
print("Bronze columns:", df.columns)

Latest ingestion_date: 2026-04-12
Bronze rows (latest): 12
Bronze columns: ['comment', 'entities_id', 'id', 'is_active', 'is_deleted', 'is_recursive', 'items_id', 'itemtype', 'links', 'ingestion_date', '_source_file', '_source_system', '_ingest_bronze_ts']


 ## Transform (rename + types + parse links)

In [0]:
df_t = (
    df
    # rename business key to match reservations join key
    .withColumn("reservation_item_id", col("id").cast("int"))
    .withColumn("reservation_item_name", col("items_id").cast("string"))
    .withColumn("entity_name", col("entities_id").cast("string"))
    .withColumn("itemtype", col("itemtype").cast("string"))
    .withColumn("comment", col("comment").cast("string"))

    # flags
    .withColumn("is_active", (col("is_active").cast("int") == lit(1)).cast("boolean"))
    .withColumn("is_deleted", (col("is_deleted").cast("int") == lit(1)).cast("boolean"))
    .withColumn("is_recursive", (col("is_recursive").cast("int") == lit(1)).cast("boolean"))

    # metadata
    .withColumn("ingestion_date", col("ingestion_date").cast("date"))
    .withColumn("_ingest_silver_ts", current_timestamp())
)

# basic quality checks (optional)
df_t = df_t.filter(col("reservation_item_id").isNotNull())

## Deduplicate by reservation_item_id (keep latest ingest)

In [0]:
w = Window.partitionBy("reservation_item_id").orderBy(col("_ingest_bronze_ts").desc_nulls_last())
df_dedup = (
    df_t
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

print("Silver reserveditems distinct ids:", df_dedup.select(countDistinct("reservation_item_id")).collect()[0][0])

Silver reserveditems distinct ids: 12


## Select final Silver schema

In [0]:
df_out = df_dedup.select(
    "reservation_item_id",
    "reservation_item_name",
    "itemtype",
    "is_active",
    "is_deleted",
    "is_recursive",
    "ingestion_date",
    "_source_file",
    "_source_system",
    "_ingest_bronze_ts",
    "_ingest_silver_ts",
)

In [0]:
df_out.printSchema()

In [0]:
df_out.display()

reservation_item_id,reservation_item_name,itemtype,is_active,is_deleted,is_recursive,ingestion_date,_source_file,_source_system,_ingest_bronze_ts,_ingest_silver_ts
1,Salle COSY (S-4),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
2,Salle Panorama (S-12),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
3,Salle Confidentielle (S-24),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
4,Salle Avant Première (S15),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
5,Salle Transparente (S-13),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
6,Meeting Room (S-23),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
7,Corner Room (S-20),PluginRoomRoom,false,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
8,Salle Arabesque (S-5),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
12,Salle 11,PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z
13,Salle 10,PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:19:24.758Z


## Write Silver (overwrite snapshot)

In [0]:
(df_out.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(silver_path)
)

print("Wrote Silver reserveditems snapshot to:", silver_path, "rows:", df_out.count())

Wrote Silver reservations snapshot to: abfss://silver@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems


## Optional validation read-back

In [0]:
df_check = spark.read.format("delta").load(silver_path)
print("Silver read-back rows:", df_check.count())

Silver read-back rows: 12


In [0]:
display(df_check)

reservation_item_id,reservation_item_name,itemtype,is_active,is_deleted,is_recursive,ingestion_date,_source_file,_source_system,_ingest_bronze_ts,_ingest_silver_ts
1,Salle COSY (S-4),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
2,Salle Panorama (S-12),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
3,Salle Confidentielle (S-24),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
4,Salle Avant Première (S15),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
5,Salle Transparente (S-13),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
6,Meeting Room (S-23),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
7,Corner Room (S-20),PluginRoomRoom,false,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
8,Salle Arabesque (S-5),PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
12,Salle 11,PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
13,Salle 10,PluginRoomRoom,true,false,false,2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:43:01.847Z,2026-04-12T17:20:36.359Z
